In [1]:
from src.models.builders import ModelBuilder
ModelBuilder.list_available()

['classifier',
 'classifier_tm_s',
 'classifier_stm_s',
 'classifier_se',
 'classifier_stm',
 'clf_attnpl',
 'clf_attnpl_t',
 'lstm',
 'thp',
 'reg_attnpl']

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.utils.mask_utils import TriangularCausalMask, ProbMask
from flash_attn.flash_attn_interface import flash_attn_varlen_qkvpacked_func
from flash_attn.bert_padding import unpad_input, pad_input
from  math import sqrt
from abc import ABC, abstractmethod
from src.utils.registrable import Registrable


class BaseAttention(nn.Module, ABC,Registrable):
    """
    Abstract base class for attention mechanisms.

    Subclasses must implement the `forward` method.

    Expected input shapes:
        query, key, value: [B, L, H, D]
        attn_mask: Optional[Tensor] with shape broadcastable to [B, H, L, S]
        padding_mask: Optional[Tensor] with shape [B, L]

    Returns:
        output: [B, L, H, D]
        attention_weights (optional): [B, H, L, S] or None
    """

    def __init__(self,output_attention=False):
        super().__init__()
        self.output_attention = output_attention

    @abstractmethod
    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        attn_mask: torch.Tensor = None,
        padding_mask: torch.Tensor = None
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Compute attention.

        Args:
            query: [B, L, H, D]
            key: [B, L, H, D]
            value: [B, L, H, D]
            attn_mask: [B, H, L, S] or [B, L, S] or None
            padding_mask: [B, L] or None

        Returns:
            output: [B, L, H, D]
            attention_weights: [B, H, L, S] or None
        """
        pass


@BaseAttention.register(name="standard")
class StandardAttention(BaseAttention):
    """
    Multi-Head Scaled Dot-Product Attention with mask support.

    Args:
        scale (float): Use 1 / sqrt(d_k) as scale.
        attn_dropout (float): Dropout rate after softmax.
        output_attention (bool): Whether to return attention weights.
    """

    def __init__(self, scale, attn_dropout=0.1, output_attention=True):
        super().__init__(output_attention=output_attention)
        self.scale = scale 
        self.dropout = nn.Dropout(attn_dropout)

    def forward(self, q, k, v, attn_mask=None, padding_mask=None):
        """
        Args:
            q, k, v: [B, L, H, D]
            attn_mask: [B, H, L, S] or [B, L, S] or None
            padding_mask: Optional, not used (reserved for flash-attn compatibility)

        Returns:
            output: [B, L, H, D]
            attn_weights (optional): [B, H, L, S]
        """
        # [B, L, H, D] → [B, H, L, D]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q , k.transpose(2, 3))  # [B, H, L, S]
        # print("Scores shape:", scores.shape)
        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)  # [B, 1, L, S]
            scores = scores.masked_fill(attn_mask, -1e-9)
        
        print("Scores shape:", scores)
        attn_weights = F.softmax(scores* self.scale, dim=-1)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, v)  # [B, H, L, D]
        output = output.transpose(1, 2)  # → [B, L, H, D]

        if self.output_attention:
            return output, attn_weights
        else:
            return output, None

@BaseAttention.register(name="full")
class FullAttention(BaseAttention):
    def __init__(self, mask_flag=True, scale=None, attn_dropout=0.1, output_attention=False):
        super().__init__(output_attention=output_attention)
        self.scale = scale
        self.mask_flag = mask_flag
        self.dropout = nn.Dropout(attn_dropout)
        
    def forward(self, q, k, v, attn_mask,padding_mask=None):
        B, L, H, E = q.shape
        _, S, _, D = v.shape
        scale = self.scale or 1./sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", q, k)
        # print("Scores shape:", scores.shape)
        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=q.device).mask

            scores.masked_fill_(attn_mask, -1e-9)
        print("Scores shape:", scores)
        attn_weights = self.dropout(torch.softmax(scale * scores, dim=-1))
        output = torch.einsum("bhls,bshd->blhd", attn_weights, v)

        if self.output_attention:
            return (output.contiguous(), attn_weights)
        else:
            return (output.contiguous(), None)

# -------------------------------
# 构造 combined attn_mask（causal + padding）
# -------------------------------
def build_combined_mask(B, H, L, S, device):
    # causal mask: [L, S]
    causal_mask = torch.triu(torch.ones(L, S, device=device), diagonal=1).bool()  # True 表示屏蔽未来
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0).expand(B, H, L, S)  # [B, H, L, S]

    # 假设 S 的最后两个位置是 padding
    padding_mask = torch.zeros(B, S, dtype=torch.bool, device=device)
    padding_mask[:, -2:] = True  # 最后两个为 padding
    padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)  # [B, 1, 1, S]
    padding_mask = padding_mask.expand(B, H, L, S)

    # 最终 mask：logical or
    final_mask = causal_mask | padding_mask
    return final_mask

# -------------------------------
# 测试函数
# -------------------------------
def test_attention_equivalence_with_padding_and_causal_mask():
    B, L, S, H, D = 2, 4, 4, 2, 8
    scale = 1.0 / sqrt(D)
    device = torch.device("cpu")
    torch.manual_seed(42)

    # 构造输入
    q = torch.randn(B, L, H, D, device=device)
    k = torch.randn(B, S, H, D, device=device)
    v = torch.randn(B, S, H, D, device=device)

    # 构造包含 padding 和 causal 的 attn_mask
    attn_mask = build_combined_mask(B, H, L, S, device)

    # 初始化模块
    standard = StandardAttention(scale=scale, attn_dropout=0.0, output_attention=True)
    ##
    full = FullAttention(mask_flag=True, scale=scale, attn_dropout=0.0, output_attention=True)
    out_std, attn_std = standard(q.clone(), k.clone(), v.clone(), attn_mask.clone())
    out_full, attn_full = full(q.clone(), k.clone(), v.clone(), attn_mask.clone())
    ##
    # full = FullAttention(mask_flag=False, scale=scale, attn_dropout=0.0, output_attention=True)
    # out_std, attn_std = standard(q.clone(), k.clone(), v.clone(), None)
    # out_full, attn_full = full(q.clone(), k.clone(), v.clone(), None)

    # 比较
    print("Standard Output shape:", out_std.shape)
    print("Full Output shape:", out_full.shape)
    print("Output一致:", torch.allclose(out_std, out_full, atol=1e-6))
    print("Attention一致:", torch.allclose(attn_std, attn_full, atol=1e-6))

test_attention_equivalence_with_padding_and_causal_mask()

Scores shape: tensor([[[[ 3.2475e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 6.5869e+00, -2.7608e+00, -1.0000e-09, -1.0000e-09],
          [-6.1588e+00,  2.5118e+00, -1.0000e-09, -1.0000e-09],
          [-2.0790e+00,  1.2847e+00, -1.0000e-09, -1.0000e-09]],

         [[-7.0944e-01, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 2.4674e+00,  5.1145e+00, -1.0000e-09, -1.0000e-09],
          [ 2.4812e+00,  3.2185e+00, -1.0000e-09, -1.0000e-09],
          [ 2.7755e+00,  2.6716e+00, -1.0000e-09, -1.0000e-09]]],


        [[[ 4.4805e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [-5.8768e-01,  1.3723e+00, -1.0000e-09, -1.0000e-09],
          [-4.8000e+00,  2.3821e+00, -1.0000e-09, -1.0000e-09],
          [-3.5210e+00,  2.9143e-01, -1.0000e-09, -1.0000e-09]],

         [[-3.2184e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 1.2057e-01,  4.0429e+00, -1.0000e-09, -1.0000e-09],
          [ 2.3836e+00, -6.3395e-01, -1.0000e-09, -1.0000e-09],
          [ 1.4165

In [4]:
float('-inf')

-inf

In [8]:
def prepare_data_lstm(args, base_dir="data/CD2021"):
    def create_lstm_data(features, target, timestep):
        """
        划分数据集，生成特征数据和目标数据
        :param features: 特征数据（二维数组），数据集的所有特征列（去除目标列）
        :param target: 目标数据（数组），数据集的目标列
        :param timestep: 时间步长，用于生成每个样本的特征序列长度
        :return: 特征数据X和目标数据Y
        """
        X, y = [], []
        
        for index in range(len(features) - timestep):
            X.append(features[index: index + timestep])
            y.append(target[index + timestep])

        # 转换为NumPy数组
        X, y = np.array(X), np.array(y)
        return X, y

    import src.data.lstm_loader as loader
    import src.features.seismic_features as sf
    from src.data.preprocessing import load_and_filter_catalog
    from src.data.data_utils import get_split_indices
    from src.utils.file_utils import save_or_load_data
    from sklearn.preprocessing import MinMaxScaler
    import numpy as np

    df = load_and_filter_catalog(base_dir, Mc=args.Mc)
    def generate_seismic_data():
        features_df, num_mag = sf.calculate_seismic_features(
            df.to_numpy(),
            Mc=args.Mc,
            Mf=args.Mf,
            Twindow=args.Twindow,
            Tfore=args.Tfore,
            dt=args.dt,
            dMag=args.dMag,
            Mag_elaps=args.Mag_elaps,
            L_max=60,
            context_len=args.context_len,
        )
        return {
            "features_df": features_df,
            "num_mag": num_mag
        }

    # 加载或生成处理过的数据
    cached_data = save_or_load_data(
        base_path=base_dir,
        generate_fn=generate_seismic_data,
        sub_dir="processed/rf_classifier",
        prefix="rf",
        Mc=args.Mc,
        Mf=args.Mf,
        Twindow=args.Twindow,
        Tfore=args.Tfore,
        dt=args.dt,
        dMag=args.dMag,
        Mag_elaps=args.Mag_elaps,
        context_len=args.context_len
    )

    features_df = cached_data["features_df"]
    features_df_nl,scalars = loader.normalize_df(features_df)
    num_mag = cached_data["num_mag"]

    features = features_df_nl[args.feature_cols].values
    target = features_df_nl['Mag_max_obs'].values.copy()
    X, y = create_lstm_data(features, target, timestep=args.time_step) 
    X,y = loader.clean_data(X,y)
    dataset,data_loaders = loader.split_dataset(
        X, y,
        by_time=args.split_by_time,
        batch_size=args.batch_size,
        train_ratio=0.8,
        val_ratio=0.1,
        time_order=getattr(args, 'time_order', ('train', 'val', 'test')),
        scalars=scalars
    )

    return features_df_nl,data_loaders['train'], data_loaders['val'], data_loaders['test'], dataset

In [12]:
args.feature_cols

['Num',
 'Mag_max',
 'Mag_mean',
 'b_lsq',
 'a_lsq',
 'b_std_lsq',
 'std_gr_lsq',
 'b_mlk',
 'a_mlk',
 'b_std_mlk',
 'std_gr_mlk',
 'dM_lsq',
 'dM_mlk',
 'Energy_sqrt',
 'prob_x7_lsq',
 'prob_x7_mlk',
 'beta',
 'zvalue']

In [3]:
from config.config_loader import load_args_from_yaml 
args= load_args_from_yaml("config/lstm.yaml")
base_dir = f"data/{args.dataset}"
df,train_loader, val_loader, test_loader,dataset = prepare_data_lstm(args, base_dir)

feature_cols: ['Num', 'Mag_max', 'Mag_mean', 'b_lsq', 'a_lsq', 'b_std_lsq', 'std_gr_lsq', 'b_mlk', 'a_mlk', 'b_std_mlk', 'std_gr_mlk', 'dM_lsq', 'dM_mlk', 'Energy_sqrt', 'prob_x7_lsq', 'prob_x7_mlk', 'beta', 'zvalue', 'T_elaps6', 'T_elaps6.5', 'T_elaps7', 'T_elaps7.5']


NameError: name 'prepare_data_lstm' is not defined

In [10]:
print(df)

            t       Num   Mag_max  Mag_max_obs  Mag_mean     b_lsq     a_lsq  \
0    0.000000  0.413616  0.942857     0.378378  0.356141  0.163697  0.432888   
1    0.001698  0.129094  0.485714     0.378378  0.264228  0.182568  0.569200   
2    0.003396  0.118176  0.400000     0.891892  0.241327  0.150467  0.625173   
3    0.005093  0.111753  0.400000     0.891892  0.200131  0.195457  0.572524   
4    0.006791  0.114965  0.400000     0.891892  0.202793  0.194969  0.583475   
..        ...       ...       ...          ...       ...       ...       ...   
585  0.993209  0.332049  0.428571     0.189189  0.248213  0.437476  0.706395   
586  0.994907  0.343610  0.428571     0.189189  0.244562  0.439897  0.718969   
587  0.996604  0.343610  0.428571     0.189189  0.237871  0.437600  0.715304   
588  0.998302  0.346821  0.428571     0.189189  0.229729  0.436310  0.712779   
589  1.000000  0.344894  0.428571     0.567568  0.227482  0.432375  0.712663   

     b_std_lsq  std_gr_lsq     b_mlk  .

In [11]:
args

namespace(model='lstm',
          task_type='regression',
          dataset='ChuanDian',
          feature_cols=['Num',
                        'Mag_max',
                        'Mag_mean',
                        'b_lsq',
                        'a_lsq',
                        'b_std_lsq',
                        'std_gr_lsq',
                        'b_mlk',
                        'a_mlk',
                        'b_std_mlk',
                        'std_gr_mlk',
                        'dM_lsq',
                        'dM_mlk',
                        'Energy_sqrt',
                        'prob_x7_lsq',
                        'prob_x7_mlk',
                        'beta',
                        'zvalue'],
          batch_size=32,
          warmup_ratio='None',
          learning_rate=0.001,
          weight_decay=1e-05,
          scheduler_factor=0.6,
          scheduler_patience=5,
          scheduler_threshold=0.0001,
          scheduler_min_lr=1e-06,
          scheduler_ty

In [1]:
import torch

def compute_nonzero_mean_per_sample(times: torch.Tensor) -> torch.Tensor:
    """
    对 times 中每个样本（行）去除为 0 的值后，计算其均值。
    
    参数：
        times (torch.Tensor): shape (batch_size, seq_len)，时间序列张量
        
    返回：
        torch.Tensor: shape (batch_size,)，每个样本的非零均值
    """
    non_zero_mask = times != 0
    times_float = times.float()
    
    sums = (times_float * non_zero_mask).sum(dim=1)
    counts = non_zero_mask.sum(dim=1).clamp(min=1)  # 防止除以0

    means = sums / counts
    return means


In [3]:
times = torch.tensor([
    [1.0, 2.0, 0.0, 3.0],
    [0.0, 0.0, 0.0, 4.0],
    [0.0, 0.0, 0.0, 0.0]
])

means = compute_nonzero_mean_per_sample(times)
print(means)
# 输出: tensor([2.0000, 4.0000, 0.0000])


tensor([2., 4., 0.])


In [13]:
times_nan = torch.tensor([
    [1.0, 2.0, float('nan'), 7.0],
    [float('nan'), float('nan'), float('nan'), 4.0],
    [float('nan'), float('nan'), float('nan'), float('nan')]
])
torch.nanquantile(times_nan,0.5,dim=1)

tensor([2., 4., nan])

In [15]:
times_nan.max(dim=1)

torch.return_types.max(
values=tensor([nan, nan, nan]),
indices=tensor([2, 0, 0]))

In [ ]:
torch.max()

In [2]:
import  torch
output = torch.load("/root/autodl-tmp/chuandian_eq/output.pt",weights_only=False)

In [7]:
output['times'][:5]

tensor([[2.0315, 2.5621, 7.2437,  ..., 0.0000, 0.0000, 0.0000],
        [2.0315, 2.5621, 7.2437,  ..., 0.0000, 0.0000, 0.0000],
        [2.0315, 2.5621, 7.2437,  ..., 0.0000, 0.0000, 0.0000],
        [2.0315, 2.5621, 7.2437,  ..., 0.0000, 0.0000, 0.0000],
        [2.0315, 2.5621, 7.2437,  ..., 0.0000, 0.0000, 0.0000]],
       device='cuda:0')

In [8]:
output['times'][-5:]

tensor([[2.0315e+00, 2.5621e+00, 7.2437e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.0315e+00, 2.5621e+00, 7.2437e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.0315e+00, 2.5621e+00, 7.2437e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.0315e+00, 2.5621e+00, 7.2437e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.0315e+00, 2.5621e+00, 7.2437e+00,  ..., 3.7540e+04, 3.7545e+04,
         0.0000e+00]], device='cuda:0')

In [1]:
import torch

def _inverse_bounded_tanh(output, min_val: float = -1, max_val: float = 1) -> torch.Tensor:
    """
    计算给定 output 对应的原始 input。
    """
    # 确保 output 值在有效范围内，避免 arctanh 出现 NaN
    # arctanh 的输入必须在 (-1, 1) 之间
    clamped_output = torch.clamp(output, min=min_val + 1e-6, max=max_val - 1e-6)

    # 1. 移项并简化
    term = (clamped_output - min_val) / (max_val - min_val) * 2 - 1
    
    # 2. 应用 arctanh
    input = torch.atanh(term)
    
    return input

# 示例
min_val = -1
max_val = 1
# 假设我们知道 output，想找回 input
known_output = torch.tensor([0.5, 0.9])

# 使用上面定义的函数来计算 input
calculated_input = _inverse_bounded_tanh(known_output, min_val, max_val)
print(f"Calculated input: {calculated_input}")

# 验证：将计算出的 input 代入原始函数，看是否能得到 known_output
# 原始函数
def _bounded_tanh(input, min_val: float = -1, max_val: float = 1) -> torch.Tensor:
    return min_val + (max_val - min_val) * 0.5 * (torch.tanh(input) + 1)

verified_output = _bounded_tanh(calculated_input, min_val, max_val)
print(f"Verified output: {verified_output}")

# 检查计算结果是否一致
assert torch.allclose(known_output, verified_output, atol=1e-5)
print("Verification successful!")

Calculated input: tensor([0.5493, 1.4722])
Verified output: tensor([0.5000, 0.9000])
Verification successful!
